
# Neural Local Volatility benchmark: Sabate/Gierjatowicz-style neural SDE on the March 2024 SPX data

This notebook builds the **neural local volatility (neural LV)** member of the benchmark in the same spirit as the repository accompanying:

> Gierjatowicz, Sabate-Vidales, Šiška, Szpruch and Žurič, *Robust Pricing and Hedging via Neural SDEs*.

The goal here is deliberately narrower than the paper's robust exotic-price-bounds experiment:

1. clean exactly the same SPX universe used by the current classical LV and Heston notebooks;
2. calibrate a neural LV model directly to the daily OTM option strip;
3. keep the **Sabate control-variate / learned-hedge mechanism inside training**;
4. obtain same-day prices;
5. freeze the day-$t$ neural local-volatility model and obtain day-$t+1$ prices for identical contracts;
6. evaluate the same one-day delta hedge used by the existing LV/Heston baselines;
7. additionally evaluate the learned Sabate control-variate hedge over that same observed one-day move.

The default `pilot` settings are designed for a laptop that will usually run PyTorch on its **CPU** when the only graphics device is Intel Iris Xe. The notebook also contains a `full` configuration for a later NVIDIA GPU or HPC run.

---

## What is kept faithful to Sabate

The following are retained because they are central to the paper/repository:

- a risk-neutral neural local-volatility SDE;
- a feed-forward volatility network with ReLU hidden layers and a positive Softplus output;
- **tamed Euler** simulation;
- direct calibration to vanilla option prices by Monte Carlo;
- a neural hedge used as a **control variate**;
- alternating optimisation: one epoch updates the SDE while the hedge is frozen, the next updates the hedge while the SDE is frozen;
- Adam with learning rate $10^{-3}$;
- gradient clipping;
- fresh Monte Carlo paths in training;
- fixed Monte Carlo paths for periodic convergence checks;
- antithetic paths in the larger final Monte Carlo evaluation.

The paper uses four hidden layers of width 50 for the SDE coefficient. The repository uses three hidden layers of width 30 for the vanilla control variate, so this notebook follows the **repository implementation** on that small paper/code discrepancy.

---

## Deliberate deviations, checked one last time

There are only adaptations needed by the real-data benchmark or by the laptop runtime.

### Deviation A: one shared $\sigma_\theta(t,x)$, not one SDE network per maturity interval

Sabate split the SDE coefficient into one network on each maturity interval and calibrated the intervals incrementally. That is elegant for their synthetic experiment with a few regular maturities. The March SPX data have roughly forty irregular expiry/settlement slices per day.

As agreed, we instead use one continuous, shared network
$$
\sigma_\theta(t,x).
$$
This loses Sabate's incremental maturity-freezing trick, but **does not change the neural-LV model class or the alternating SDE/control-variate training idea**.

### Deviation B: forward-normalised state coordinates

Sabate's synthetic experiment starts at $S_0=1$. We use
$$
X_u=\frac{S_u}{F_0(u)},
$$
where $F_0(u)$ is the observed deterministic forward curve on the calibration date. Then
$$
dX_u=X_u\sigma_\theta(u,X_u)\,dW_u,
\qquad X_0=1.
$$

This is a change of coordinates, not a different pricing model. It:

- preserves the convenient $X_0=1$ scaling;
- removes deterministic carry from the simulated state;
- still uses the actual OptionMetrics forward and discount curves.

For a contract with maturity $T$,
$$
C_0=D(0,T)F_0(T)
\,\mathbb E\!\left[\left(X_T-\frac{K}{F_0(T)}\right)^+\right]
$$
for a call, with the analogous put payoff.

### Deviation C: the control-variate network is contract-conditioned

Sabate's repository has one output neuron for every calibration option. That is natural for $21$ strikes times a few maturities, but it becomes awkward with thousands of irregular SPX contracts and it cannot produce a learned hedge for contracts outside the exact calibration subset.

We therefore use one shared hedge network
$$
h_\xi(t,X;T,k,\mathrm{call/put},\mathrm{AM/PM}),
\qquad k=\log(K/F_0(T)).
$$

**The control-variate philosophy is unchanged.** The network still learns a predictable underlying-stock hedge by minimising pathwise estimator variance. The change is only how the many individual hedge functions are parametrised.

### Deviation D: laptop-sized Monte Carlo and contract mini-batches

The repository uses $40{,}000$ simulated paths per SGD iteration, 20 iterations per epoch, up to 1000 epochs, and 400,000 paths for evaluation. That is not realistic on an Intel-Iris-Xe laptop because normal PyTorch CUDA acceleration requires an NVIDIA GPU.

`pilot` therefore reduces path counts, option counts and epochs. `full` restores Sabate-scale Monte Carlo choices as far as is sensible for this real-data experiment.

### Deviation E: no exotic lookback branch

The repository simultaneously carries a lookback-option control variate because the paper studies robust bounds for illiquid exotic claims. This benchmark is currently about vanilla pricing and one-day hedging, so the unrelated exotic branch is omitted.

Everything else below is kept as close as possible to the repository while respecting the existing LV/Heston benchmark conventions.



## 1. Imports, device detection and reproducibility

**ETA** means *estimated time remaining*. `tqdm` uses the speed of completed iterations to display an ETA automatically.

For an Intel Iris Xe laptop, standard PyTorch will usually print `cpu` below. If a future environment provides a supported Intel `xpu` backend, the notebook will use it; an NVIDIA machine will use CUDA.


In [2]:

from pathlib import Path
from dataclasses import dataclass
import copy
import json
import math
import os
import random
import warnings

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")  # safer non-interactive backend for VS Code / Windows
import matplotlib.pyplot as plt
from IPython.display import Image as IPythonImage

from scipy.interpolate import PchipInterpolator
from tqdm.auto import tqdm

import torch
import torch.nn as nn

warnings.filterwarnings("ignore", category=FutureWarning)

SEED = 12345
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

def choose_device():
    if torch.cuda.is_available():
        return torch.device("cuda")
    if hasattr(torch, "xpu"):
        try:
            if torch.xpu.is_available():
                return torch.device("xpu")
        except Exception:
            pass
    return torch.device("cpu")

DEVICE = choose_device()
print("PyTorch version:", torch.__version__)
print("Selected device:", DEVICE)
print("CPU logical cores:", os.cpu_count())

if DEVICE.type == "cpu" and os.cpu_count():
    torch.set_num_threads(max(1, min(8, os.cpu_count())))
    print("PyTorch CPU threads:", torch.get_num_threads())


PyTorch version: 2.9.1+cpu
Selected device: cpu
CPU logical cores: 8
PyTorch CPU threads: 8



## 2. Runtime configurations

Start with `pilot`.

- `smoke` is only for checking that every cell works.
- `pilot` is intended to obtain a genuine first result on the laptop.
- `full` is the later research run for an NVIDIA GPU or HPC node.

The **method** is the same across modes. What changes is Monte Carlo accuracy and how much of the option strip is visited per update.


In [3]:

RUN_MODE = "pilot"   # "smoke", "pilot", or "full"

MODE = {
    "smoke": {
        "max_quotes_per_slice": 3,
        "epochs": 4,
        "paths_per_step": 64,
        "contracts_per_step": 24,
        "steps_per_epoch": 1,
        "fixed_eval_paths": 256,
        "fixed_eval_contracts": 24,
        "eval_every": 2,
        "final_mc_paths": 2_000,
        "mc_path_chunk": 1_000,
        "contract_chunk": 24,
        "patience_checks": None,
    },
    "pilot": {
        "max_quotes_per_slice": 9,
        "epochs": 80,
        "paths_per_step": 512,
        "contracts_per_step": 96,
        "steps_per_epoch": 4,
        "fixed_eval_paths": 2_048,
        "fixed_eval_contracts": 80,
        "eval_every": 10,
        "final_mc_paths": 20_000,
        "mc_path_chunk": 5_000,
        "contract_chunk": 64,
        "patience_checks": 5,
    },
    "full": {
        "max_quotes_per_slice": 25,
        "epochs": 1_000,
        "paths_per_step": 40_000,
        "contracts_per_step": 128,  # real-data contract mini-batch; all-at-once is too large
        "steps_per_epoch": 20,
        "fixed_eval_paths": 40_000,
        "fixed_eval_contracts": 200,
        "eval_every": 10,
        "final_mc_paths": 400_000,
        "mc_path_chunk": 20_000,
        "contract_chunk": 128,
        "patience_checks": None,
    },
}[RUN_MODE]

SDE_HIDDEN_LAYERS = 4
SDE_WIDTH = 50
HEDGE_HIDDEN_LAYERS = 3
HEDGE_WIDTH = 30
LEARNING_RATE = 1e-3
SDE_GRAD_CLIP = 5.0
HEDGE_GRAD_CLIP = 3.0
BASE_STEPS_PER_YEAR = 96
TARGET_NORMALISED_RMSE = 2e-5
DELTA_BUMP_SPX_POINTS = 0.50

EXAMPLE_DATE = "2024-03-01"
FORCE_RETRAIN = False
RUN_OPTIONAL_ROLLING_MONTH = False
WARM_START_SIGMA_ACROSS_DAYS = False

CACHE_DIR = Path("neural_lv_sabate_cache")
CACHE_DIR.mkdir(exist_ok=True)

print(json.dumps(MODE, indent=2))


{
  "max_quotes_per_slice": 9,
  "epochs": 80,
  "paths_per_step": 512,
  "contracts_per_step": 96,
  "steps_per_epoch": 4,
  "fixed_eval_paths": 2048,
  "fixed_eval_contracts": 80,
  "eval_every": 10,
  "final_mc_paths": 20000,
  "mc_path_chunk": 5000,
  "contract_chunk": 64,
  "patience_checks": 5
}



## 3. Locate and load the four OptionMetrics/IvyDB files

The notebook first looks in the current directory and then in `/mnt/data` (the ChatGPT working directory used when this notebook was built).


In [4]:

def find_file(pattern):
    search_dirs = [Path("."), Path("/mnt/data")]
    found = []
    for folder in search_dirs:
        found.extend(sorted(folder.glob(pattern)))
    if not found:
        raise FileNotFoundError(f"Could not find a file matching {pattern!r}")
    return found[0]

FILES = {
    "options": find_file("one_month_spx_option_price.csv"),
    "forwards": find_file("one_month_spx_forward_price.csv"),
    "spot": find_file("one_month_spx_security_price.csv"),
    "rates": find_file("one_month_spx_zero_coupon_yield_curve.csv"),
}

for name, path in FILES.items():
    print(f"{name:8s}: {path}")

options_raw = pd.read_csv(FILES["options"], low_memory=False)
forwards_raw = pd.read_csv(FILES["forwards"], low_memory=False)
spot_raw = pd.read_csv(FILES["spot"], low_memory=False)
rates_raw = pd.read_csv(FILES["rates"], low_memory=False)

print({
    "options": len(options_raw),
    "forwards": len(forwards_raw),
    "spot": len(spot_raw),
    "rates": len(rates_raw),
})


options : one_month_spx_option_price.csv
forwards: one_month_spx_forward_price.csv
spot    : one_month_spx_security_price.csv
rates   : one_month_spx_zero_coupon_yield_curve.csv
{'options': 326964, 'forwards': 1192, 'spot': 20, 'rates': 220}



## 4. Exact common cleaning universe from the current LV/Heston notebooks

This section is intentionally copied from the shared LV/Heston cleaning logic:

- SPX (`secid = 108105`);
- European options;
- standard 100-multiplier, unadjusted contracts;
- calls and puts;
- $7$ to $365$ calendar days to expiry;
- positive, non-crossed bid/ask;
- IvyDB implied volatility in a numerical sanity range;
- $0.05\le |\Delta_{\rm IvyDB}|\le 0.95$;
- relative spread at most $25\%$;
- $|\log(K/F)|\le 0.30$;
- elementary European price bounds.

For calibration, calls and puts are reduced to one **OTM observation per strike**. Near ATM, the tighter-spread side is preferred. A maturity/settlement slice must contain at least eight OTM observations.

The complete cleaned call-and-put chain is retained for same-day and one-day out-of-sample evaluation.


In [5]:

SECID = 108105
MIN_DTE, MAX_DTE = 7, 365
MAX_ABS_LOG_MONEYNESS = 0.30
MIN_ABS_DELTA, MAX_ABS_DELTA = 0.05, 0.95
MAX_RELATIVE_SPREAD = 0.25
ATM_TIGHTER_SIDE_BAND = 0.05
MIN_POINTS_PER_SLICE = 8

def prepare_clean_data(options_raw, forwards_raw, spot_raw, rates_raw):
    options = options_raw.copy()
    options["date"] = pd.to_datetime(options["date"])
    options["exdate"] = pd.to_datetime(options["exdate"])
    options["K"] = options["strike_price"] / 1000.0
    options["T_days"] = (options["exdate"] - options["date"]).dt.days
    options["T"] = options["T_days"] / 365.0
    options["bid"] = pd.to_numeric(options["best_bid"], errors="coerce")
    options["ask"] = pd.to_numeric(options["best_offer"], errors="coerce")
    options["mid"] = 0.5 * (options["bid"] + options["ask"])
    options["spread"] = options["ask"] - options["bid"]
    options["relative_spread"] = options["spread"] / options["mid"].replace(0, np.nan)
    options["iv_market"] = pd.to_numeric(options["impl_volatility"], errors="coerce")
    options["delta_market"] = pd.to_numeric(options["delta"], errors="coerce")
    options["abs_delta_market"] = options["delta_market"].abs()

    mask = (
        (options["secid"] == SECID)
        & (options["exercise_style"] == "E")
        & (options["contract_size"] == 100)
        & (options["cfadj"] == 1)
        & options["cp_flag"].isin(["C", "P"])
        & (options["bid"] >= 0)
        & (options["ask"] > options["bid"])
        & (options["mid"] > 0)
        & options["T_days"].between(MIN_DTE, MAX_DTE)
        & options["iv_market"].between(1e-4, 5.0)
        & options["abs_delta_market"].between(MIN_ABS_DELTA, MAX_ABS_DELTA)
        & (options["relative_spread"] <= MAX_RELATIVE_SPREAD)
    )
    options = options.loc[mask].copy()

    spot = spot_raw.loc[spot_raw["secid"] == SECID, ["date", "close"]].copy()
    spot["date"] = pd.to_datetime(spot["date"])
    spot = spot.rename(columns={"close": "S"}).drop_duplicates("date")

    forwards = forwards_raw.loc[forwards_raw["secid"] == SECID].copy()
    forwards["date"] = pd.to_datetime(forwards["date"])
    forwards["expiration"] = pd.to_datetime(forwards["expiration"])
    forwards = forwards.rename(columns={
        "expiration": "exdate",
        "AMSettlement": "am_settlement",
        "ForwardPrice": "F",
    })
    forwards = forwards[
        ["date", "exdate", "am_settlement", "F"]
    ].drop_duplicates(["date", "exdate", "am_settlement"])

    rates = rates_raw.copy()
    rates["date"] = pd.to_datetime(rates["date"])
    rates["days"] = pd.to_numeric(rates["days"], errors="coerce")
    rates["rate_decimal"] = pd.to_numeric(rates["rate"], errors="coerce") / 100.0
    rate_curves = {
        d: (g["days"].to_numpy(float), g["rate_decimal"].to_numpy(float))
        for d, g in rates.dropna().sort_values(["date", "days"]).groupby("date")
    }

    options = options.merge(spot, on="date", how="left").merge(
        forwards,
        on=["date", "exdate", "am_settlement"],
        how="left",
        validate="many_to_one",
    )
    options["r"] = [
        float(np.interp(n, *rate_curves[d])) if d in rate_curves else np.nan
        for d, n in zip(options["date"], options["T_days"])
    ]
    options["D"] = np.exp(-options["r"] * options["T"])
    options["carry"] = np.log(options["F"] / options["S"]) / options["T"]
    options["q"] = options["r"] - options["carry"]
    options["k"] = np.log(options["K"] / options["F"])

    options = options.dropna(subset=["S", "F", "r", "D", "carry", "k"])
    options = options.loc[options["k"].abs() <= MAX_ABS_LOG_MONEYNESS].copy()

    call_lb = options["D"] * np.maximum(options["F"] - options["K"], 0.0)
    call_ub = options["D"] * options["F"]
    put_lb = options["D"] * np.maximum(options["K"] - options["F"], 0.0)
    put_ub = options["D"] * options["K"]

    lb = np.where(options["cp_flag"].eq("C"), call_lb, put_lb)
    ub = np.where(options["cp_flag"].eq("C"), call_ub, put_ub)
    options = options.loc[
        (options["mid"] >= lb - 1e-6)
        & (options["mid"] <= ub + 1e-6)
    ].copy()

    return options.sort_values(
        ["date", "exdate", "am_settlement", "K", "cp_flag"]
    ).reset_index(drop=True)

def build_otm_surface(quotes):
    idx = [
        "date", "exdate", "am_settlement", "K", "T", "S", "F",
        "r", "q", "carry", "D", "k"
    ]
    vals = [
        "mid", "bid", "ask", "spread", "relative_spread",
        "iv_market", "delta_market", "optionid"
    ]

    wide = quotes.pivot_table(
        index=idx, columns="cp_flag", values=vals, aggfunc="first"
    )
    wide.columns = [f"{a}_{b}" for a, b in wide.columns]
    wide = wide.reset_index()

    strict_call = wide["K"] >= wide["F"]
    near = wide["k"].abs() <= ATM_TIGHTER_SIDE_BAND

    have_c = wide.get("iv_market_C", pd.Series(np.nan, index=wide.index)).notna()
    have_p = wide.get("iv_market_P", pd.Series(np.nan, index=wide.index)).notna()
    c_spread = wide.get("spread_C", pd.Series(np.inf, index=wide.index))
    p_spread = wide.get("spread_P", pd.Series(np.inf, index=wide.index))

    choose_call = strict_call.copy()
    both_near = near & have_c & have_p
    choose_call = np.where(both_near, c_spread <= p_spread, choose_call)
    wide["cp_otm"] = np.where(choose_call, "C", "P")

    for base in vals:
        c = wide.get(f"{base}_C", pd.Series(np.nan, index=wide.index))
        p = wide.get(f"{base}_P", pd.Series(np.nan, index=wide.index))
        wide[f"{base}_otm"] = np.where(choose_call, c, p)

    wide = wide.dropna(subset=["iv_market_otm", "mid_otm"]).copy()

    sizes = wide.groupby(["date", "exdate", "am_settlement"]).size()
    valid = sizes[sizes >= MIN_POINTS_PER_SLICE].index

    return (
        wide.set_index(["date", "exdate", "am_settlement"])
        .loc[valid]
        .reset_index()
        .sort_values(["date", "T", "k"])
        .reset_index(drop=True)
    )

quotes_all = prepare_clean_data(options_raw, forwards_raw, spot_raw, rates_raw)
otm = build_otm_surface(quotes_all)

print("Complete cleaned quotes:", len(quotes_all))
print("OTM calibration observations:", len(otm))
print("Trading dates:", otm["date"].nunique())
print(
    "Expiry/settlement slices:",
    otm.groupby(["date", "exdate", "am_settlement"]).ngroups,
)

daily_counts = pd.DataFrame({
    "clean_quotes": quotes_all.groupby("date").size(),
    "otm_quotes": otm.groupby("date").size(),
    "slices": otm.groupby("date").apply(
        lambda g: g.groupby(["exdate", "am_settlement"]).ngroups
    ),
})
display(daily_counts)


Complete cleaned quotes: 192031
OTM calibration observations: 91930
Trading dates: 20
Expiry/settlement slices: 809


C:\Users\Startklar\AppData\Local\Temp\ipykernel_51204\3433349330.py:161: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  "slices": otm.groupby("date").apply(


,clean_quotes,otm_quotes,slices
date,,,
2024-03-01,9046,4304,40
2024-03-04,9361,4502,40
2024-03-05,9619,4658,40
2024-03-06,9571,4619,40
2024-03-07,9304,4483,40
2024-03-08,9713,4664,40
2024-03-11,9625,4662,40
2024-03-12,9264,4360,40
2024-03-13,9284,4384,39



## 5. Deterministic market curves

The classical LV/Heston notebooks use the observed OptionMetrics forward curve and zero-coupon curve rather than imposing one constant interest rate.

The same `DailyMarketCurve` idea is used here.

For later one-day hedging we need:
$$
\int_t^{t+1}q(u)\,du
$$
and the one-day cash-account growth. These are therefore computed exactly as in the existing baselines.


In [6]:

@dataclass
class DailyMarketCurve:
    date: pd.Timestamp
    am_settlement: int
    spot: float
    T_nodes: np.ndarray
    logD_nodes: np.ndarray
    logF_nodes: np.ndarray

    def __post_init__(self):
        self._logD = PchipInterpolator(
            self.T_nodes, self.logD_nodes, extrapolate=True
        )
        self._logF = PchipInterpolator(
            self.T_nodes, self.logF_nodes, extrapolate=True
        )

    def log_discount(self, t):
        return self._logD(np.maximum(np.asarray(t, float), 0.0))

    def log_forward_ratio(self, t):
        return self._logF(np.maximum(np.asarray(t, float), 0.0))

    def discount(self, t0, t1):
        return np.exp(self.log_discount(t1) - self.log_discount(t0))

    def forward(self, t):
        return self.spot * np.exp(self.log_forward_ratio(t))

    def q_integral(self, t0, t1):
        return (
            -(self.log_discount(t1) - self.log_discount(t0))
            -(self.log_forward_ratio(t1) - self.log_forward_ratio(t0))
        )

class CurveBook:
    def __init__(self, spot_raw, forwards_raw, rates_raw):
        self.spot = spot_raw.loc[
            spot_raw["secid"] == SECID, ["date", "close"]
        ].copy()
        self.spot["date"] = pd.to_datetime(self.spot["date"])
        self.spot = self.spot.set_index("date")["close"].to_dict()

        self.forwards = forwards_raw.loc[forwards_raw["secid"] == SECID].copy()
        self.forwards["date"] = pd.to_datetime(self.forwards["date"])
        self.forwards["expiration"] = pd.to_datetime(self.forwards["expiration"])

        self.rates = rates_raw.copy()
        self.rates["date"] = pd.to_datetime(self.rates["date"])
        self.rates["rate_decimal"] = pd.to_numeric(
            self.rates["rate"], errors="coerce"
        ) / 100.0

        self._cache = {}

    def get(self, date, am_settlement):
        date = pd.Timestamp(date)
        am = int(am_settlement)
        key = (date, am)

        if key in self._cache:
            return self._cache[key]

        S0 = float(self.spot[date])

        rg = self.rates.loc[
            self.rates["date"] == date,
            ["days", "rate_decimal"],
        ].dropna().sort_values("days")

        T_r = rg["days"].to_numpy(float) / 365.0
        logD = -rg["rate_decimal"].to_numpy(float) * T_r

        fg = self.forwards.loc[
            (self.forwards["date"] == date)
            & (self.forwards["AMSettlement"] == am),
            ["expiration", "ForwardPrice"],
        ].copy()

        if len(fg) < 2:
            fg = self.forwards.loc[
                self.forwards["date"] == date,
                ["expiration", "ForwardPrice"],
            ].copy()

        fg["T"] = (fg["expiration"] - date).dt.days / 365.0
        fg = (
            fg.loc[(fg["T"] >= 0) & (fg["ForwardPrice"] > 0)]
            .groupby("T", as_index=False)["ForwardPrice"]
            .mean()
            .sort_values("T")
        )

        T_f = fg["T"].to_numpy(float)
        logF = np.log(fg["ForwardPrice"].to_numpy(float) / S0)

        positive = T_f > 0
        T_f, logF = T_f[positive], logF[positive]

        T_all = np.unique(np.concatenate([[0.0], T_r, T_f]))
        logD_all = np.interp(T_all, np.r_[0.0, T_r], np.r_[0.0, logD])
        logF_all = np.interp(T_all, np.r_[0.0, T_f], np.r_[0.0, logF])

        curve = DailyMarketCurve(
            date, am, S0, T_all, logD_all, logF_all
        )
        self._cache[key] = curve
        return curve

curve_book = CurveBook(spot_raw, forwards_raw, rates_raw)



## 6. Neural LV and the Sabate control variate

### 6.1 Neural local volatility

In the paper:
$$
dS_t=rS_t\,dt+S_t\sigma_\theta(t,S_t)\,dW_t.
$$

In this notebook's forward-normalised coordinate:
$$
dX_t=X_t\sigma_\theta(t,X_t)\,dW_t.
$$

The volatility network takes only $(t,X_t)$, matching the Markovian neural-LV idea.

### 6.2 Why the control variate remains central

For a dividend-paying underlying under the risk-neutral measure,
$$
dS_t=(r_t-q_t)S_tdt+S_t\sigma_t dW_t.
$$

The stochastic part of the **discounted cum-dividend gain** is
$$
D(0,t)S_t\sigma_t\,dW_t.
$$

Therefore, if $h_\xi$ is interpreted as a number of index units, the zero-mean trading control variate is approximated by
$$
\sum_k
h_\xi(t_k,\ldots)\,
D(0,t_k)S_{t_k}\sigma_{t_k}\Delta W_k.
$$

This is the dividend-aware analogue of the repository line
```python
exp(-r*t) * S_old * diffusion * hedge * dW
```

and the learned $h_\xi(0,\ldots)$ can be compared with the model delta as an actual underlying hedge ratio.

### 6.3 The crucial Sabate detach

When updating $\theta$, Sabate feeds **detached** state/diffusion values to the hedge/control-variate calculation. The control variate changes the noisy Monte Carlo estimate used inside the squared pricing loss, but its derivative is not allowed to create an extra gradient path through the hedge.

We reproduce that logic below.


In [7]:

class FeedForwardNet(nn.Module):
    def __init__(
        self,
        input_dim,
        output_dim,
        hidden_layers,
        width,
        output_activation="identity",
    ):
        super().__init__()
        layers = []
        in_dim = input_dim

        for _ in range(hidden_layers):
            layers.append(nn.Linear(in_dim, width))
            layers.append(nn.ReLU())
            in_dim = width

        layers.append(nn.Linear(in_dim, output_dim))

        if output_activation == "softplus":
            layers.append(nn.Softplus())
        elif output_activation != "identity":
            raise ValueError(output_activation)

        self.net = nn.Sequential(*layers)

    def forward(self, x):
        return self.net(x)

def sabate_xavier_initialisation(module):
    if isinstance(module, nn.Linear):
        nn.init.xavier_normal_(module.weight.data, gain=1.5)

class NeuralLocalVol(nn.Module):
    """Shared-time version of Sabate's neural LV diffusion."""
    def __init__(self):
        super().__init__()
        self.net = FeedForwardNet(
            input_dim=2,
            output_dim=1,
            hidden_layers=SDE_HIDDEN_LAYERS,
            width=SDE_WIDTH,
            output_activation="softplus",
        )

    def forward(self, t, x):
        if t.ndim == 0:
            t = torch.full_like(x, float(t.item()))
        elif t.ndim == 1:
            t = t[:, None]
        if x.ndim == 1:
            x = x[:, None]
        return self.net(torch.cat([t, x], dim=1))

class ContractConditionedHedge(nn.Module):
    """One shared real-data hedge network producing stock units."""
    def __init__(self):
        super().__init__()
        self.net = FeedForwardNet(
            input_dim=6,
            output_dim=1,
            hidden_layers=HEDGE_HIDDEN_LAYERS,
            width=HEDGE_WIDTH,
            output_activation="identity",
        )

    def forward(self, features):
        original_shape = features.shape[:-1]
        y = self.net(features.reshape(-1, features.shape[-1]))
        return y.reshape(*original_shape)

def set_requires_grad(module, value):
    for p in module.parameters():
        p.requires_grad_(value)



## 7. Calibration strip and simulation time grid

The full cleaned OTM universe remains the common data universe. For neural-SDE optimisation we use a **deterministic spread over each expiry/settlement slice**, just as the Heston notebook caps the number of quotes per slice.

`pilot` keeps 9 points per slice. `full` keeps 25.

This is not a second cleaning rule: it is only a computationally manageable calibration subset selected from the already-clean OTM surface.

### Time discretisation

Sabate uses 96 uniform steps over one year. Here we start from the same $1/96$-year spacing and insert every observed maturity exactly into the grid. This prevents an actual listed expiry from being approximated by the nearest simulation time.


In [8]:

def sample_calibration_day(day, max_per_slice):
    pieces = []
    for _, g in day.groupby(["exdate", "am_settlement"]):
        g = g.sort_values("k")
        if len(g) <= max_per_slice:
            pieces.append(g)
        else:
            positions = np.unique(
                np.round(
                    np.linspace(0, len(g) - 1, max_per_slice)
                ).astype(int)
            )
            pieces.append(g.iloc[positions])
    return pd.concat(pieces, ignore_index=True)

def build_time_grid(maturities, steps_per_year=BASE_STEPS_PER_YEAR):
    maturities = np.asarray(maturities, float)
    max_T = float(np.max(maturities))
    base = np.arange(
        0.0, max_T + 0.5 / steps_per_year, 1.0 / steps_per_year
    )
    base = base[base <= max_T + 1e-12]
    grid = np.unique(np.r_[0.0, base, maturities, max_T])
    grid.sort()
    return grid

def nearest_grid_indices(grid, times):
    grid = np.asarray(grid)
    times = np.asarray(times)
    idx = np.searchsorted(grid, times)
    idx = np.clip(idx, 0, len(grid) - 1)
    left = np.maximum(idx - 1, 0)
    choose_left = np.abs(grid[left] - times) < np.abs(grid[idx] - times)
    idx = np.where(choose_left, left, idx)

    err = np.max(np.abs(grid[idx] - times))
    if err > 1e-10:
        raise RuntimeError(f"Maturity not represented exactly. max error={err}")
    return idx

@dataclass
class CalibrationContext:
    date: pd.Timestamp
    spot: float
    grid_np: np.ndarray
    grid: torch.Tensor
    contracts: pd.DataFrame

    maturity_index: torch.Tensor
    K: torch.Tensor
    T: torch.Tensor
    F_T: torch.Tensor
    D_T: torch.Tensor
    target: torch.Tensor
    k: torch.Tensor
    cp_sign: torch.Tensor
    am: torch.Tensor

    F_grid: torch.Tensor
    D_grid: torch.Tensor

def build_calibration_context(date, max_quotes_per_slice):
    date = pd.Timestamp(date)
    day = otm.loc[otm["date"] == date].copy()

    if day.empty:
        raise ValueError(f"No OTM data for {date.date()}")

    calib = sample_calibration_day(day, max_quotes_per_slice)
    calib = calib.rename(columns={
        "mid_otm": "target_mid",
        "cp_otm": "target_cp",
    }).reset_index(drop=True)

    maturities = np.sort(calib["T"].unique())
    grid_np = build_time_grid(maturities)
    maturity_index = nearest_grid_indices(
        grid_np, calib["T"].to_numpy(float)
    )
    S0 = float(calib["S"].iloc[0])

    F_grid_np = np.empty((2, len(grid_np)))
    D_grid_np = np.empty((2, len(grid_np)))

    for am in [0, 1]:
        curve = curve_book.get(date, am)
        F_grid_np[am] = curve.forward(grid_np)
        D_grid_np[am] = curve.discount(0.0, grid_np)

    cp_sign = np.where(calib["target_cp"].eq("C"), 1.0, -1.0)

    return CalibrationContext(
        date=date,
        spot=S0,
        grid_np=grid_np,
        grid=torch.tensor(grid_np, dtype=torch.float32, device=DEVICE),
        contracts=calib,
        maturity_index=torch.tensor(
            maturity_index, dtype=torch.long, device=DEVICE
        ),
        K=torch.tensor(calib["K"].to_numpy(float), dtype=torch.float32, device=DEVICE),
        T=torch.tensor(calib["T"].to_numpy(float), dtype=torch.float32, device=DEVICE),
        F_T=torch.tensor(calib["F"].to_numpy(float), dtype=torch.float32, device=DEVICE),
        D_T=torch.tensor(calib["D"].to_numpy(float), dtype=torch.float32, device=DEVICE),
        target=torch.tensor(
            calib["target_mid"].to_numpy(float), dtype=torch.float32, device=DEVICE
        ),
        k=torch.tensor(calib["k"].to_numpy(float), dtype=torch.float32, device=DEVICE),
        cp_sign=torch.tensor(cp_sign, dtype=torch.float32, device=DEVICE),
        am=torch.tensor(
            calib["am_settlement"].to_numpy(int), dtype=torch.long, device=DEVICE
        ),
        F_grid=torch.tensor(F_grid_np, dtype=torch.float32, device=DEVICE),
        D_grid=torch.tensor(D_grid_np, dtype=torch.float32, device=DEVICE),
    )



## 8. One Sabate-style Monte Carlo training step

For selected option contracts and fresh Brownian paths:

1. simulate the neural LV state with tamed Euler;
2. accumulate the learned trading control variate until each contract's maturity;
3. form the pathwise estimator
   $$
   \widehat\Phi_i^{cv}
   =
   D_i\Phi_i(S_{T_i})
   -
   \sum_{k<T_i}
   h_{\xi,i}(t_k,\ldots)\,
   D_kS_k\sigma_k\Delta W_k;
   $$
4. average over Monte Carlo paths to obtain the option-price estimate.

### SDE-update epoch

The hedge is frozen. State and volatility values entering the control variate are detached, reproducing Sabate's key gradient treatment.

### Hedge-update epoch

The SDE is frozen. We minimise the sample variance of the pathwise control-variate estimators.

The loss is divided by $S_0^2$ only for numerical scaling. Because $S_0$ is one constant for every option on a given day, this does **not** reweight contracts or change the minimiser relative to dollar-price squared error.


In [9]:

def static_contract_features(ctx, ids):
    return torch.stack([
        ctx.T[ids],
        ctx.k[ids],
        ctx.cp_sign[ids],
        ctx.am[ids].float(),
    ], dim=1)

def make_hedge_features(t_value, x_state, static_features):
    B = x_state.shape[0]
    M = static_features.shape[0]

    t = torch.full(
        (B, M, 1),
        float(t_value),
        dtype=x_state.dtype,
        device=x_state.device,
    )
    x = x_state[:, None, :].expand(B, M, 1)
    static = static_features[None, :, :].expand(B, M, 4)
    return torch.cat([t, x, static], dim=2)

def pathwise_cv_estimators(model, hedge, ctx, contract_ids, z, phase):
    ids = torch.as_tensor(contract_ids, dtype=torch.long, device=DEVICE)

    maturity_idx = ctx.maturity_index[ids]
    K = ctx.K[ids]
    F_T = ctx.F_T[ids]
    D_T = ctx.D_T[ids]
    cp = ctx.cp_sign[ids]
    am = ctx.am[ids]
    static = static_contract_features(ctx, ids)

    B = z.shape[0]
    M = len(ids)

    x = torch.ones((B, 1), dtype=torch.float32, device=DEVICE)
    cv = torch.zeros((B, M), dtype=torch.float32, device=DEVICE)
    payoff = torch.zeros((B, M), dtype=torch.float32, device=DEVICE)

    max_step = int(maturity_idx.max().item())

    for step in range(max_step):
        t = ctx.grid[step]
        dt = ctx.grid[step + 1] - ctx.grid[step]
        sqrt_dt = torch.sqrt(dt)

        if phase == "hedge":
            with torch.no_grad():
                sigma = model(t, x)
        else:
            sigma = model(t, x)

        dW = sqrt_dt * z[:, step:step + 1]

        active = torch.where(maturity_idx > step)[0]

        if len(active):
            features = make_hedge_features(
                float(t.item()), x.detach(), static[active]
            )

            if phase == "sde":
                with torch.no_grad():
                    h = hedge(features)
            else:
                h = hedge(features)

            stochastic_piece = x.detach() * sigma.detach() * dW
            am_active = am[active]
            discounted_forward = (
                ctx.D_grid[am_active, step] * ctx.F_grid[am_active, step]
            )
            contribution = (
                h
                * stochastic_piece
                * discounted_forward[None, :]
            )
            cv = cv.index_add(1, active, contribution)

        a = x * sigma
        denominator = (
            1.0 + torch.abs(x.detach() * sigma.detach()) * sqrt_dt
        )
        x = x + a * dW / denominator

        matured = torch.where(maturity_idx == step + 1)[0]

        if len(matured):
            S_T = x * F_T[matured][None, :]
            strike = K[matured][None, :]
            call_payoff = torch.clamp(S_T - strike, min=0.0)
            put_payoff = torch.clamp(strike - S_T, min=0.0)
            is_call = (cp[matured] > 0)[None, :]
            discounted = (
                torch.where(is_call, call_payoff, put_payoff)
                * D_T[matured][None, :]
            )
            payoff = payoff.index_add(1, matured, discounted)

    estimator = payoff - cv
    model_price = estimator.mean(dim=0)
    return estimator, model_price

def draw_contract_ids(n_contracts, batch_size):
    if batch_size is None or batch_size >= n_contracts:
        return np.arange(n_contracts)
    return np.random.choice(
        n_contracts, size=batch_size, replace=False
    )

def antithetic_normals(total_paths, n_steps, seed):
    if total_paths < 2:
        raise ValueError("Need at least 2 paths.")

    half = (total_paths + 1) // 2
    gen = torch.Generator(device="cpu")
    gen.manual_seed(seed)

    z = torch.randn(
        half,
        n_steps,
        generator=gen,
        dtype=torch.float32,
    )
    z = torch.cat([z, -z], dim=0)[:total_paths]
    return z.to(DEVICE)



## 9. Fixed Monte Carlo evaluation paths during training

These are **not held-out market options**.

Training uses fresh Brownian paths, so the training loss can move merely because different random numbers were drawn. We therefore create a modest set of Brownian paths **once** and reuse exactly those paths every few epochs.

That gives a more stable answer to:

> Is the neural SDE genuinely fitting better, or did this epoch simply receive lucky Monte Carlo noise?

The actual day-$t+1$ option quotes remain the economic out-of-sample test.

Sabate's repository uses 200,000 Brownian paths plus their 200,000 antithetic partners for evaluation. `pilot` uses far fewer only because this laptop is expected to run on CPU.


In [10]:

def evenly_spread_contract_ids(contracts, n):
    if n is None or n >= len(contracts):
        return np.arange(len(contracts))

    ordered = contracts.sort_values(["T", "k"]).index.to_numpy()
    positions = np.unique(
        np.round(np.linspace(0, len(ordered) - 1, n)).astype(int)
    )
    return ordered[positions]

@torch.no_grad()
def evaluate_fixed_subset(
    model, hedge, ctx, eval_ids, z_eval, contract_chunk
):
    prices = []
    targets = []

    for start in range(0, len(eval_ids), contract_chunk):
        ids = eval_ids[start:start + contract_chunk]
        _, price = pathwise_cv_estimators(
            model, hedge, ctx, ids, z_eval, phase="sde"
        )

        prices.append(price.detach().cpu())
        ids_t = torch.as_tensor(ids, dtype=torch.long, device=DEVICE)
        targets.append(ctx.target[ids_t].detach().cpu())

    price = torch.cat(prices).numpy()
    target = torch.cat(targets).numpy()
    error = price - target

    return {
        "normalised_rmse": np.sqrt(np.mean(error**2)) / ctx.spot,
        "dollar_rmse": np.sqrt(np.mean(error**2)),
        "dollar_mae": np.mean(np.abs(error)),
    }



## 10. Alternating Sabate-style training

The progress bar has one entry per **epoch**.

- odd-numbered human-readable epochs (`epoch=1,3,5,...`) update the SDE;
- even-numbered epochs update the control-variate hedge.

The progress bar reports the latest fixed-path RMSE when that check is run and automatically displays elapsed time and ETA.

The repository's learning-rate schedule only changes after epochs 500 and 800. We retain that schedule; it therefore has no effect in `pilot`, which intentionally stops earlier.


In [11]:

def checkpoint_path(date):
    date = pd.Timestamp(date)
    return CACHE_DIR / f"neural_lv_{date:%Y%m%d}_{RUN_MODE}.pt"

def train_neural_lv(date, initial_sigma_state=None):
    ctx = build_calibration_context(date, MODE["max_quotes_per_slice"])

    model = NeuralLocalVol().to(DEVICE)
    hedge = ContractConditionedHedge().to(DEVICE)

    model.apply(sabate_xavier_initialisation)
    hedge.apply(sabate_xavier_initialisation)

    if initial_sigma_state is not None:
        model.load_state_dict(initial_sigma_state)

    optimizer_sde = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)
    optimizer_hedge = torch.optim.Adam(hedge.parameters(), lr=LEARNING_RATE)

    scheduler_sde = torch.optim.lr_scheduler.MultiStepLR(
        optimizer_sde, milestones=[500, 800], gamma=0.2
    )

    n_contracts = len(ctx.contracts)

    eval_ids = evenly_spread_contract_ids(
        ctx.contracts, MODE["fixed_eval_contracts"]
    )
    z_eval = antithetic_normals(
        MODE["fixed_eval_paths"],
        len(ctx.grid_np) - 1,
        SEED + 777,
    )

    history = []
    best_metric = np.inf
    best_model_state = None
    best_hedge_state = None
    checks_without_improvement = 0

    epoch_bar = tqdm(
        range(MODE["epochs"]),
        desc=f"Training neural LV {pd.Timestamp(date).date()}",
        unit="epoch",
    )

    latest_eval = {
        "normalised_rmse": np.nan,
        "dollar_rmse": np.nan,
    }
    stop = False

    for epoch in epoch_bar:
        update_hedge = ((epoch + 1) % 2 == 0)
        phase = "hedge" if update_hedge else "sde"

        set_requires_grad(model, not update_hedge)
        set_requires_grad(hedge, update_hedge)

        step_losses = []

        for _ in range(MODE["steps_per_epoch"]):
            ids = draw_contract_ids(
                n_contracts, MODE["contracts_per_step"]
            )

            z = torch.randn(
                MODE["paths_per_step"],
                len(ctx.grid_np) - 1,
                dtype=torch.float32,
                device=DEVICE,
            )

            optimizer_sde.zero_grad(set_to_none=True)
            optimizer_hedge.zero_grad(set_to_none=True)

            estimator, price = pathwise_cv_estimators(
                model, hedge, ctx, ids, z, phase=phase
            )

            ids_t = torch.as_tensor(ids, dtype=torch.long, device=DEVICE)

            if phase == "sde":
                target = ctx.target[ids_t]
                loss = torch.mean(((price - target) / ctx.spot) ** 2)
                loss.backward()
                nn.utils.clip_grad_norm_(
                    model.parameters(), SDE_GRAD_CLIP
                )
                optimizer_sde.step()
            else:
                scaled = estimator / ctx.spot
                loss = torch.var(
                    scaled, dim=0, unbiased=False
                ).mean()
                loss.backward()
                nn.utils.clip_grad_norm_(
                    hedge.parameters(), HEDGE_GRAD_CLIP
                )
                optimizer_hedge.step()

            step_losses.append(float(loss.detach().cpu()))

        scheduler_sde.step()

        do_eval = (
            epoch == 0
            or (epoch + 1) % MODE["eval_every"] == 0
            or epoch == MODE["epochs"] - 1
        )

        if do_eval:
            set_requires_grad(model, False)
            set_requires_grad(hedge, False)

            latest_eval = evaluate_fixed_subset(
                model,
                hedge,
                ctx,
                eval_ids,
                z_eval,
                MODE["contract_chunk"],
            )

            metric = latest_eval["normalised_rmse"]

            if metric < best_metric:
                best_metric = metric
                best_model_state = copy.deepcopy(model.state_dict())
                best_hedge_state = copy.deepcopy(hedge.state_dict())
                checks_without_improvement = 0
            else:
                checks_without_improvement += 1

            if metric < TARGET_NORMALISED_RMSE:
                print("Reached Sabate-style normalised RMSE target.")
                stop = True

            patience = MODE["patience_checks"]
            if (
                patience is not None
                and checks_without_improvement >= patience
            ):
                print(
                    "Pilot early stopping: fixed-path RMSE has not improved "
                    f"for {patience} checks."
                )
                stop = True

        history.append({
            "epoch": epoch + 1,
            "phase": phase,
            "train_loss": np.mean(step_losses),
            "eval_normalised_rmse": latest_eval["normalised_rmse"],
            "eval_dollar_rmse": latest_eval["dollar_rmse"],
        })

        epoch_bar.set_postfix({
            "phase": phase,
            "loss": f"{np.mean(step_losses):.2e}",
            "eval_$RMSE": (
                "NA"
                if np.isnan(latest_eval["dollar_rmse"])
                else f"{latest_eval['dollar_rmse']:.3f}"
            ),
        })

        if stop:
            break

    if best_model_state is not None:
        model.load_state_dict(best_model_state)
        hedge.load_state_dict(best_hedge_state)

    set_requires_grad(model, False)
    set_requires_grad(hedge, False)
    model.eval()
    hedge.eval()

    return model, hedge, ctx, pd.DataFrame(history)

def train_or_load_neural_lv(date, initial_sigma_state=None):
    path = checkpoint_path(date)

    if path.exists() and not FORCE_RETRAIN:
        payload = torch.load(path, map_location=DEVICE, weights_only=False)

        model = NeuralLocalVol().to(DEVICE)
        hedge = ContractConditionedHedge().to(DEVICE)

        model.load_state_dict(payload["model_state"])
        hedge.load_state_dict(payload["hedge_state"])

        model.eval()
        hedge.eval()
        set_requires_grad(model, False)
        set_requires_grad(hedge, False)

        ctx = build_calibration_context(
            date, payload["max_quotes_per_slice"]
        )
        history = pd.DataFrame(payload["history"])

        print("Loaded checkpoint:", path)
        return model, hedge, ctx, history

    model, hedge, ctx, history = train_neural_lv(
        date, initial_sigma_state
    )

    torch.save(
        {
            "date": str(pd.Timestamp(date).date()),
            "run_mode": RUN_MODE,
            "max_quotes_per_slice": MODE["max_quotes_per_slice"],
            "model_state": model.state_dict(),
            "hedge_state": hedge.state_dict(),
            "history": history.to_dict("records"),
        },
        path,
    )
    print("Saved checkpoint:", path)

    return model, hedge, ctx, history



## 11. Train the first calibration date

The first date in the supplied month is Friday 1 March 2024, so its next trading observation is Monday 4 March. The out-of-sample step is therefore three **calendar** days, exactly as in the existing LV/Heston notebooks.

A `pilot` run can still take meaningful time on CPU. The progress bar's ETA will become more stable after several epochs.


In [12]:

calibration_date = pd.Timestamp(EXAMPLE_DATE)

available_dates = sorted(
    pd.to_datetime(quotes_all["date"].unique())
)

if calibration_date not in available_dates:
    raise ValueError(f"{calibration_date.date()} is not in the data.")

date_position = available_dates.index(calibration_date)

if date_position == len(available_dates) - 1:
    raise ValueError("Need a later trading date for the next-day test.")

next_date = available_dates[date_position + 1]

print("Calibration date:", calibration_date.date())
print("Next trading date:", next_date.date())
print("Calendar-day gap:", (next_date - calibration_date).days)

model, hedge, calib_ctx, history = train_or_load_neural_lv(
    calibration_date
)

display(history.tail(20))


Calibration date: 2024-03-01
Next trading date: 2024-03-04
Calendar-day gap: 3
Loaded checkpoint: neural_lv_sabate_cache\neural_lv_20240301_pilot.pt


,epoch,phase,train_loss,eval_normalised_rmse,eval_dollar_rmse
60,61,sde,0.000007,0.002536,13.028474
61,62,hedge,0.000390,0.002536,13.028474
62,63,sde,0.000007,0.002536,13.028474
63,64,hedge,0.000392,0.002536,13.028474
64,65,sde,0.000006,0.002536,13.028474
65,66,hedge,0.000389,0.002536,13.028474
66,67,sde,0.000006,0.002536,13.028474
67,68,hedge,0.000396,0.002536,13.028474
68,69,sde,0.000007,0.002536,13.028474
69,70,hedge,0.000435,0.002124,10.911286



## 12. Training diagnostics: safe version

Section 11 has already produced/loaded the trained `model`, `hedge`, `calib_ctx` and `history`. Nothing in the pricing or hedging benchmark depends on making these plots.

On the original run, the kernel died in this diagnostic section without a Python traceback. Because the calculations here are tiny, that is much more consistent with a native plotting/backend failure (or a stale kernel after a long run) than with the neural-LV calculation itself.

This version therefore does two things:

1. **numerical diagnostics first**, with no plotting at all;
2. optional figures are rendered with Matplotlib's non-interactive `Agg` backend, saved to PNG, then displayed as files. We deliberately avoid `plt.show()`.

The fixed-path RMSE is only evaluated at epoch 1 and every `eval_every` epochs. Repeated values in the Section 11 table between those epochs do **not** mean the fit was re-evaluated and stayed exactly flat; the notebook was simply carrying forward the most recent diagnostic value.


In [ ]:

# ---- 12A. Numerical diagnostics only: no plotting ----

if not len(history):
    raise RuntimeError("Training history is empty.")

# Keep only the epochs on which the fixed Monte Carlo diagnostic was actually run.
actual_check = (
    (history["epoch"] == 1)
    | (history["epoch"] % MODE["eval_every"] == 0)
    | (history["epoch"] == history["epoch"].max())
)

diagnostic_history = history.loc[
    actual_check,
    [
        "epoch",
        "phase",
        "train_loss",
        "eval_normalised_rmse",
        "eval_dollar_rmse",
    ],
].copy()

print("Fixed-path evaluation checks:")
display(diagnostic_history)

best_row = diagnostic_history.loc[
    diagnostic_history["eval_dollar_rmse"].idxmin()
]

print(
    f"\nBest fixed-path RMSE: ${best_row['eval_dollar_rmse']:.4f} "
    f"at epoch {int(best_row['epoch'])}"
)
print(
    f"Normalised RMSE at that check: "
    f"{best_row['eval_normalised_rmse']:.6f}"
)
print(
    "\nReminder: this is a Monte Carlo calibration diagnostic on the fixed "
    "evaluation subset, not the final complete-chain same-day RMSE and not "
    "the next-day out-of-sample result."
)

# Very small, safe neural-LV evaluation.
t_grid_plot = np.linspace(
    0.0,
    min(1.0, float(calib_ctx.grid_np[-1])),
    8,
)
x_grid_plot = np.linspace(0.75, 1.25, 101)

sigma_rows = []

with torch.inference_mode():
    for t in t_grid_plot:
        x_t = torch.tensor(
            x_grid_plot,
            dtype=torch.float32,
            device=DEVICE,
        )[:, None]
        t_t = torch.full_like(x_t, float(t))

        sig = (
            model(t_t, x_t)
            .squeeze(1)
            .detach()
            .cpu()
            .numpy()
        )

        if not np.all(np.isfinite(sig)):
            raise RuntimeError(
                f"Non-finite local-volatility values found at t={t:.4f}."
            )

        sigma_rows.append(sig)

sigma_array = np.vstack(sigma_rows)

print("\nLearned local-volatility diagnostic:")
print("shape:", sigma_array.shape)
print("minimum sigma:", float(sigma_array.min()))
print("median sigma:", float(np.median(sigma_array)))
print("maximum sigma:", float(sigma_array.max()))

# If this cell completes, the model evaluation itself is healthy.
print(
    "\nSection 12A completed successfully. "
    "You can proceed directly to Section 13 even if you do not make the plots."
)



### 12B. Optional diagnostic figures

This cell is optional. It saves figures to PNG using the non-interactive backend instead of calling `plt.show()`.

If your VS Code kernel still crashes here, leave `MAKE_DIAGNOSTIC_PLOTS = False` and continue to Section 13. That would isolate the problem to local plotting/rendering rather than the neural-LV model.


In [ ]:

MAKE_DIAGNOSTIC_PLOTS = False  # switch to True only after Section 12A succeeds

if MAKE_DIAGNOSTIC_PLOTS:
    plot_dir = CACHE_DIR / "diagnostic_plots"
    plot_dir.mkdir(exist_ok=True)

    # Convergence plot
    fig, ax = plt.subplots(figsize=(8, 4))
    ax.plot(
        diagnostic_history["epoch"],
        diagnostic_history["eval_dollar_rmse"],
        marker="o",
    )
    ax.set_xlabel("Epoch")
    ax.set_ylabel("Fixed-path calibration RMSE ($)")
    ax.set_title(f"Neural LV convergence: {calibration_date.date()}")
    ax.grid(alpha=0.25)

    convergence_path = plot_dir / (
        f"neural_lv_convergence_{calibration_date:%Y%m%d}.png"
    )
    fig.savefig(convergence_path, dpi=140, bbox_inches="tight")
    plt.close(fig)

    display(IPythonImage(filename=str(convergence_path)))

    # Learned local-volatility slices
    fig, ax = plt.subplots(figsize=(8, 5))

    for t, sig in zip(t_grid_plot, sigma_rows):
        ax.plot(
            x_grid_plot,
            sig,
            label=f"t={t:.2f}",
        )

    ax.set_xlabel("Forward-normalised state X")
    ax.set_ylabel("Learned local volatility")
    ax.set_title("Neural LV diffusion slices")
    ax.legend(ncol=2, fontsize=8)
    ax.grid(alpha=0.25)

    sigma_path = plot_dir / (
        f"neural_lv_sigma_slices_{calibration_date:%Y%m%d}.png"
    )
    fig.savefig(sigma_path, dpi=140, bbox_inches="tight")
    plt.close(fig)

    display(IPythonImage(filename=str(sigma_path)))

    print("Saved:", convergence_path)
    print("Saved:", sigma_path)
else:
    print(
        "Optional plots skipped. This does not affect any later pricing "
        "or hedging result."
    )



## 13. Large Monte Carlo evaluation after training

During training we used modest fixed evaluation paths only to track convergence.

Now we run a larger, antithetic Monte Carlo simulation **once**.

For the full cleaned chain, raw Monte Carlo prices can be computed very efficiently from the empirical terminal distribution. The control variate is still central because it was used in training, and we separately diagnose its variance reduction below. The final full-chain pricing routine avoids evaluating a hedge network at every time step for roughly ten thousand contracts, which would dominate laptop runtime.

For same-day model delta we use **common-random-number bump-and-revalue**. A spot bump keeps the day-$t$ carry curve $F_0(T)/S_0$ fixed, exactly as a conventional spot bump keeps deterministic rates/dividends fixed. Thus the bumped forward curve is scaled by $S_0^{\pm}/S_0$, while the learned neural local-volatility function and the simulated Brownian shocks are unchanged.

This is the appropriate finite-difference spot delta in the forward-normalised coordinate: the normalised state still starts at $X_0=1$, but the absolute forward scale and therefore the option payoff move with spot.


In [ ]:

def tkey(t):
    return round(float(t), 12)

def make_eval_grid(maturities):
    return build_time_grid(maturities, BASE_STEPS_PER_YEAR)

@torch.no_grad()
def simulate_same_day_x_samples(
    model,
    maturities,
    total_paths,
    chunk_paths,
    seed,
):
    """Simulate the base forward-normalised state X, with X_0=1.

    Spot delta does not require a second SDE simulation: under a spot
    bump with deterministic carry held fixed, the absolute forward curve
    scales with spot while the normalised X dynamics and Brownian paths
    remain unchanged.
    """
    grid = make_eval_grid(maturities)
    maturity_idx = {
        tkey(T): int(nearest_grid_indices(grid, [T])[0])
        for T in np.unique(np.asarray(maturities, float))
    }

    collected = {key: [] for key in maturity_idx}
    index_to_keys = {}
    for key, idx in maturity_idx.items():
        index_to_keys.setdefault(idx, []).append(key)

    generator = torch.Generator(device="cpu")
    generator.manual_seed(seed)

    done = 0
    bar = tqdm(
        total=total_paths,
        desc="Final same-day MC",
        unit="path",
    )

    while done < total_paths:
        n = min(chunk_paths, total_paths - done)

        half = (n + 1) // 2
        z_half = torch.randn(
            half,
            len(grid) - 1,
            generator=generator,
            dtype=torch.float32,
        )
        z = torch.cat([z_half, -z_half], dim=0)[:n].to(DEVICE)

        x = torch.ones((n, 1), dtype=torch.float32, device=DEVICE)

        for step in range(len(grid) - 1):
            t = float(grid[step])
            dt = float(grid[step + 1] - grid[step])
            sqrt_dt = math.sqrt(dt)

            t_tensor = torch.full_like(x, t)
            sigma = model(t_tensor, x)

            dW = sqrt_dt * z[:, step:step + 1]
            a = x * sigma
            denominator = 1.0 + torch.abs(x * sigma) * sqrt_dt
            x = x + a * dW / denominator

            idx = step + 1
            if idx in index_to_keys:
                arr = x.squeeze(1).detach().cpu().numpy()
                for key in index_to_keys[idx]:
                    collected[key].append(arr.copy())

        done += n
        bar.update(n)

    bar.close()

    return {
        key: np.concatenate(pieces)
        for key, pieces in collected.items()
    }

def empirical_payoff_mean(samples, strike_norm, cp_flag):
    x = np.sort(np.asarray(samples, float))
    n = len(x)

    prefix = np.r_[0.0, np.cumsum(x)]
    total = prefix[-1]

    k = np.asarray(strike_norm, float)
    idx = np.searchsorted(x, k, side="right")

    below_sum = prefix[idx]
    above_sum = total - below_sum

    call = (above_sum - k * (n - idx)) / n
    put = (k * idx - below_sum) / n

    return np.where(np.asarray(cp_flag) == "C", call, put)

def price_same_day_from_samples(day_quotes, samples_by_T, spot, bump):
    out = day_quotes.copy()

    base_price = np.empty(len(out))
    up_price = np.empty(len(out))
    down_price = np.empty(len(out))

    for T, idx in out.groupby("T").groups.items():
        pos = out.index.get_indexer(idx)
        sample = samples_by_T[tkey(T)]

        g = out.loc[idx]
        S0 = float(spot)
        spot_levels = [S0, S0 + bump, S0 - bump]

        for spot_level, target in zip(
            spot_levels, [base_price, up_price, down_price]
        ):
            forward_scale = spot_level / S0
            F_bumped = g["F"].to_numpy(float) * forward_scale
            strike_norm_bumped = g["K"].to_numpy(float) / F_bumped

            payoff = empirical_payoff_mean(
                sample,
                strike_norm_bumped,
                g["cp_flag"].to_numpy(),
            )
            target[pos] = (
                g["D"].to_numpy(float)
                * F_bumped
                * payoff
            )

    out["neural_lv_price"] = base_price
    out["neural_lv_delta"] = (
        up_price - down_price
    ) / (2.0 * bump)

    out["pricing_error"] = out["neural_lv_price"] - out["mid"]
    out["inside_spread"] = out["neural_lv_price"].between(
        out["bid"], out["ask"]
    )

    return out



## 14. Same-day pricing on the complete cleaned call-and-put chain


In [ ]:

day0_quotes = quotes_all.loc[
    quotes_all["date"] == calibration_date
].copy().reset_index(drop=True)

same_day_samples = simulate_same_day_x_samples(
    model=model,
    maturities=day0_quotes["T"].unique(),
    total_paths=MODE["final_mc_paths"],
    chunk_paths=MODE["mc_path_chunk"],
    seed=SEED + 2000,
)

same_day_results = price_same_day_from_samples(
    day0_quotes,
    same_day_samples,
    spot=float(day0_quotes["S"].iloc[0]),
    bump=DELTA_BUMP_SPX_POINTS,
)

same_day_summary = pd.Series({
    "n": len(same_day_results),
    "RMSE": np.sqrt(np.mean(same_day_results["pricing_error"] ** 2)),
    "MAE": same_day_results["pricing_error"].abs().mean(),
    "inside_bid_ask": same_day_results["inside_spread"].mean(),
})

display(same_day_summary.to_frame("neural_LV_same_day"))

display(
    same_day_results[
        [
            "optionid", "cp_flag", "K", "T",
            "mid", "neural_lv_price",
            "neural_lv_delta", "pricing_error",
        ]
    ].head(20)
)



## 15. A direct control-variate variance-reduction diagnostic

The full-chain price table above uses the large raw Monte Carlo distribution for speed.

To verify that the Sabate control variate is doing what it should, we now take a manageable subset of the **actual calibration contracts** and compare:

$$
\operatorname{Var}(D_T\Phi)
\quad\text{against}\quad
\operatorname{Var}(D_T\Phi-\mathrm{CV}).
$$

A variance ratio below one means the learned hedge is reducing Monte Carlo variance.


In [ ]:

@torch.no_grad()
def control_variate_diagnostic(
    model,
    hedge,
    ctx,
    n_contracts=24,
    n_paths=2_000,
):
    ids = evenly_spread_contract_ids(
        ctx.contracts,
        min(n_contracts, len(ctx.contracts)),
    )

    z = antithetic_normals(
        n_paths,
        len(ctx.grid_np) - 1,
        SEED + 3000,
    )

    estimator, _ = pathwise_cv_estimators(
        model, hedge, ctx, ids, z, phase="sde"
    )

    zero_hedge = copy.deepcopy(hedge)
    for p in zero_hedge.parameters():
        p.data.zero_()

    raw_estimator, _ = pathwise_cv_estimators(
        model, zero_hedge, ctx, ids, z, phase="sde"
    )

    raw_var = torch.var(
        raw_estimator, dim=0, unbiased=False
    ).cpu().numpy()

    cv_var = torch.var(
        estimator, dim=0, unbiased=False
    ).cpu().numpy()

    rows = ctx.contracts.loc[
        ids,
        [
            "exdate", "am_settlement",
            "K", "target_cp", "target_mid",
        ],
    ].copy()

    rows["raw_variance"] = raw_var
    rows["cv_variance"] = cv_var
    rows["variance_ratio"] = (
        cv_var / np.maximum(raw_var, 1e-16)
    )

    return rows

cv_diagnostic = control_variate_diagnostic(
    model, hedge, calib_ctx
)

display(cv_diagnostic)

print(
    "Median CV/raw variance ratio:",
    cv_diagnostic["variance_ratio"].median(),
)



## 16. Identical-contract next-day pairs

This is copied conceptually from the LV/Heston notebooks.

An option enters the one-day benchmark only if the exact same `optionid`, call/put flag, strike, expiration and settlement convention are observed on both consecutive trading dates.


In [ ]:

def build_evaluation_pairs(quotes):
    trading_dates = sorted(quotes["date"].unique())
    next_map = {
        trading_dates[i]: trading_dates[i + 1]
        for i in range(len(trading_dates) - 1)
    }

    left = quotes.copy()
    left["next_date"] = left["date"].map(next_map)

    cols = [
        "date", "optionid", "cp_flag",
        "K", "exdate", "am_settlement",
        "mid", "bid", "ask",
        "S", "F", "D", "r", "q",
        "carry", "T", "k",
    ]

    right = quotes[cols].rename(columns={
        "date": "next_date",
        "mid": "mid_next",
        "bid": "bid_next",
        "ask": "ask_next",
        "S": "S_next",
        "F": "F_next",
        "D": "D_next",
        "r": "r_next",
        "q": "q_next",
        "carry": "carry_next",
        "T": "T_next",
        "k": "k_next",
    })

    return left.merge(
        right,
        on=[
            "next_date", "optionid",
            "cp_flag", "K", "exdate",
            "am_settlement",
        ],
        how="inner",
        validate="many_to_one",
    )

evaluation_pairs = build_evaluation_pairs(quotes_all)

transition_pairs = (
    evaluation_pairs.loc[
        (evaluation_pairs["date"] == calibration_date)
        & (evaluation_pairs["next_date"] == next_date)
    ]
    .copy()
    .reset_index(drop=True)
)

print("Identical-contract pairs:", len(transition_pairs))



## 17. Frozen neural-LV next-day pricing

This is the most important dynamic convention.

The day-$t$ neural local-volatility function is **not recalibrated**.

For day $t+1$:

- use the new spot;
- use the new forward/discount curve;
- use the shorter remaining maturity;
- advance the neural model's absolute clock by the calendar gap;
- evaluate the frozen day-$t$ volatility function in the **day-$t$ calibration coordinate**.

For a settlement convention $a$, define the day-$t+1$ forward-normalised state
$$
Y_v=\frac{S_{t+1+v}}{F_{t+1}^{(a)}(v)}.
$$

The old model sees
$$
X_{\rm old}(h+v)
=
\frac{S_{t+1+v}}
     {F_t^{(a)}(h+v)}
=
\frac{F_{t+1}^{(a)}(v)}
     {F_t^{(a)}(h+v)}
Y_v,
$$
where $h$ is the elapsed calendar time from calibration date to next trading date.

Therefore
$$
dY_v
=
Y_v
\sigma_\theta\!\left(
h+v,
\frac{F_{t+1}^{(a)}(v)}
     {F_t^{(a)}(h+v)}
Y_v
\right)dW_v.
$$

This is the Monte Carlo analogue of the existing classical-LV notebook's `time_origin` shift and frozen calibration curve.


In [ ]:

@torch.no_grad()
def simulate_next_day_y_samples(
    model,
    curve0,
    curve1,
    elapsed,
    maturities_remaining,
    total_paths,
    chunk_paths,
    seed,
):
    grid = make_eval_grid(maturities_remaining)

    maturity_idx = {
        tkey(T): int(nearest_grid_indices(grid, [T])[0])
        for T in np.unique(np.asarray(maturities_remaining, float))
    }

    index_to_keys = {}
    for key, idx in maturity_idx.items():
        index_to_keys.setdefault(idx, []).append(key)

    collected = {key: [] for key in maturity_idx}

    generator = torch.Generator(device="cpu")
    generator.manual_seed(seed)

    done = 0
    bar = tqdm(
        total=total_paths,
        desc=f"Next-day MC AM={curve1.am_settlement}",
        unit="path",
    )

    while done < total_paths:
        n = min(chunk_paths, total_paths - done)

        half = (n + 1) // 2
        z_half = torch.randn(
            half,
            len(grid) - 1,
            generator=generator,
            dtype=torch.float32,
        )
        z = torch.cat([z_half, -z_half], dim=0)[:n].to(DEVICE)

        y = torch.ones(
            (n, 1),
            dtype=torch.float32,
            device=DEVICE,
        )

        for step in range(len(grid) - 1):
            v = float(grid[step])
            dt = float(grid[step + 1] - grid[step])
            sqrt_dt = math.sqrt(dt)

            abs_t = elapsed + v

            F_new = float(curve1.forward(v))
            F_old = float(curve0.forward(abs_t))

            x_old = y * (F_new / F_old)

            t_tensor = torch.full_like(y, abs_t)
            sigma = model(t_tensor, x_old)

            dW = sqrt_dt * z[:, step:step + 1]
            a = y * sigma
            denominator = 1.0 + torch.abs(y * sigma) * sqrt_dt
            y = y + a * dW / denominator

            idx = step + 1
            if idx in index_to_keys:
                arr = y.squeeze(1).detach().cpu().numpy()
                for key in index_to_keys[idx]:
                    collected[key].append(arr.copy())

        done += n
        bar.update(n)

    bar.close()

    return {
        key: np.concatenate(pieces)
        for key, pieces in collected.items()
    }

def price_next_day_from_samples(pairs, samples_by_am):
    out = pairs.copy()
    price = np.empty(len(out))

    for (am, T), idx in out.groupby(
        ["am_settlement", "T_next"]
    ).groups.items():
        pos = out.index.get_indexer(idx)
        g = out.loc[idx]

        samples = samples_by_am[int(am)][tkey(T)]

        strike_norm = (
            g["K"].to_numpy(float)
            / g["F_next"].to_numpy(float)
        )

        payoff = empirical_payoff_mean(
            samples,
            strike_norm,
            g["cp_flag"].to_numpy(),
        )

        price[pos] = (
            g["D_next"].to_numpy(float)
            * g["F_next"].to_numpy(float)
            * payoff
        )

    out["model_price_next"] = price
    out["next_day_pricing_error"] = (
        out["model_price_next"] - out["mid_next"]
    )
    out["next_day_inside_bid_ask"] = (
        out["model_price_next"].between(
            out["bid_next"], out["ask_next"]
        )
    )

    return out


In [ ]:

elapsed = (next_date - calibration_date).days / 365.0

samples_next_by_am = {}

for am in sorted(transition_pairs["am_settlement"].unique()):
    g = transition_pairs.loc[
        transition_pairs["am_settlement"] == am
    ]

    curve0 = curve_book.get(calibration_date, int(am))
    curve1 = curve_book.get(next_date, int(am))

    samples_next_by_am[int(am)] = simulate_next_day_y_samples(
        model=model,
        curve0=curve0,
        curve1=curve1,
        elapsed=elapsed,
        maturities_remaining=g["T_next"].unique(),
        total_paths=MODE["final_mc_paths"],
        chunk_paths=MODE["mc_path_chunk"],
        seed=SEED + 4000 + int(am),
    )

next_day_results = price_next_day_from_samples(
    transition_pairs, samples_next_by_am
)

next_day_summary = pd.Series({
    "n": len(next_day_results),
    "RMSE": np.sqrt(
        np.mean(next_day_results["next_day_pricing_error"] ** 2)
    ),
    "MAE": next_day_results["next_day_pricing_error"].abs().mean(),
    "inside_bid_ask": next_day_results["next_day_inside_bid_ask"].mean(),
})

display(next_day_summary.to_frame("neural_LV_next_day"))



## 18. Two one-day hedge ratios

### Headline hedge: model delta

This is the apples-to-apples benchmark against BS/LV/Heston.

$$
\Delta_t
\approx
\frac{
V_t(S_t+0.5)-V_t(S_t-0.5)
}{1.0}.
$$

The day-$t$ local-volatility function and Brownian shocks are held fixed across the bump.

### Additional Sabate-native hedge

The learned control-variate network is evaluated at
$$
t=0,\qquad X_0=1.
$$

Because it was trained as the coefficient multiplying the discounted underlying gain, its output is already interpreted as **units of the underlying index**.

It is evaluated over the exact same observed one-day hedge P&L equation. It is additional evidence, not a replacement for the common delta benchmark.


In [ ]:

@torch.no_grad()
def learned_hedge_at_time_zero(hedge, rows):
    T = torch.tensor(
        rows["T"].to_numpy(float),
        dtype=torch.float32,
        device=DEVICE,
    )
    k = torch.tensor(
        rows["k"].to_numpy(float),
        dtype=torch.float32,
        device=DEVICE,
    )
    cp = torch.tensor(
        np.where(rows["cp_flag"].eq("C"), 1.0, -1.0),
        dtype=torch.float32,
        device=DEVICE,
    )
    am = torch.tensor(
        rows["am_settlement"].to_numpy(float),
        dtype=torch.float32,
        device=DEVICE,
    )

    features = torch.stack([
        torch.zeros_like(T),
        torch.ones_like(T),
        T,
        k,
        cp,
        am,
    ], dim=1)

    return hedge(features).squeeze(-1).cpu().numpy()

delta_lookup = same_day_results[
    ["optionid", "neural_lv_delta"]
].drop_duplicates("optionid")

hedge_results = next_day_results.merge(
    delta_lookup,
    on="optionid",
    how="left",
    validate="many_to_one",
)

hedge_results["learned_cv_hedge"] = learned_hedge_at_time_zero(
    hedge, hedge_results
)

def one_day_hedging_error(rows, hedge_ratio):
    rows = rows.copy().reset_index(drop=True)
    hedge_ratio = np.asarray(hedge_ratio, float)

    q_integral = np.empty(len(rows))
    cash_interest = np.empty(len(rows))

    dt = (
        rows["next_date"].iloc[0]
        - rows["date"].iloc[0]
    ).days / 365.0

    for am, idx in rows.groupby("am_settlement").groups.items():
        curve0 = curve_book.get(
            rows.loc[idx, "date"].iloc[0],
            int(am),
        )

        pos = np.asarray(list(idx), dtype=int)

        q_integral[pos] = float(
            curve0.q_integral(0.0, dt)
        )

        cash0 = (
            rows.loc[idx, "mid"].to_numpy(float)
            - hedge_ratio[pos]
            * rows.loc[idx, "S"].to_numpy(float)
        )

        cash_interest[pos] = (
            cash0
            * (
                1.0
                / float(curve0.discount(0.0, dt))
                - 1.0
            )
        )

    dividend_income = (
        hedge_ratio
        * rows["S"].to_numpy(float)
        * q_integral
    )

    return (
        rows["mid_next"].to_numpy(float)
        - rows["mid"].to_numpy(float)
        - hedge_ratio
        * (
            rows["S_next"].to_numpy(float)
            - rows["S"].to_numpy(float)
        )
        - dividend_income
        - cash_interest
    )

hedge_results["delta_hedging_error"] = one_day_hedging_error(
    hedge_results,
    hedge_results["neural_lv_delta"],
)

hedge_results["cv_hedging_error"] = one_day_hedging_error(
    hedge_results,
    hedge_results["learned_cv_hedge"],
)

hedge_results["unhedged_change"] = (
    hedge_results["mid_next"] - hedge_results["mid"]
)



## 19. Final one-transition benchmark summary


In [ ]:

def hedge_metric_summary(error, unhedged):
    error = np.asarray(error, float)
    unhedged = np.asarray(unhedged, float)

    mae = np.mean(np.abs(error))
    unhedged_mae = np.mean(np.abs(unhedged))

    return {
        "hedging_RMSE": np.sqrt(np.mean(error**2)),
        "hedging_MAE": mae,
        "hedging_bias": np.mean(error),
        "unhedged_MAE": unhedged_mae,
        "hedging_improvement": 1.0 - mae / unhedged_mae,
    }

summary = {
    "same_day_complete_chain": {
        "n": len(same_day_results),
        "pricing_RMSE": np.sqrt(
            np.mean(same_day_results["pricing_error"] ** 2)
        ),
        "pricing_MAE": same_day_results["pricing_error"].abs().mean(),
        "inside_bid_ask": same_day_results["inside_spread"].mean(),
    },
    "next_day_identical_contracts": {
        "n": len(hedge_results),
        "pricing_RMSE": np.sqrt(
            np.mean(hedge_results["next_day_pricing_error"] ** 2)
        ),
        "pricing_MAE": hedge_results["next_day_pricing_error"].abs().mean(),
        "inside_bid_ask": hedge_results["next_day_inside_bid_ask"].mean(),
    },
    "one_day_model_delta_hedge": {
        "n": len(hedge_results),
        **hedge_metric_summary(
            hedge_results["delta_hedging_error"],
            hedge_results["unhedged_change"],
        ),
    },
    "one_day_learned_CV_hedge": {
        "n": len(hedge_results),
        **hedge_metric_summary(
            hedge_results["cv_hedging_error"],
            hedge_results["unhedged_change"],
        ),
    },
}

summary_table = pd.DataFrame(summary).T
display(summary_table)

print(
    "Correlation between model delta and learned CV hedge:",
    np.corrcoef(
        hedge_results["neural_lv_delta"],
        hedge_results["learned_cv_hedge"],
    )[0, 1],
)

display(
    hedge_results[
        [
            "optionid", "cp_flag", "K", "T",
            "neural_lv_delta", "learned_cv_hedge",
            "delta_hedging_error", "cv_hedging_error",
            "next_day_pricing_error",
        ]
    ].head(30)
)



## 20. Optional: rolling the same experiment across the month

The notebook deliberately defaults to **one complete transition first**. That is the right debugging order for a neural SDE.

Once the 1 March $\to$ 4 March result looks numerically sensible, the same functions can be rolled across the month.

### Important methodological choice: warm starts

The paper explicitly treats neural-network initialisation as part of model uncertainty. Therefore the first benchmark cycle should **not silently warm-start and then pretend initialisation is irrelevant**.

`WARM_START_SIGMA_ACROSS_DAYS=False` is the default.

Later, warm-starting the previous day's $\sigma_\theta$ is a very sensible runtime/rolling-calibration ablation, but it should be labelled as such.


In [ ]:

def run_optional_rolling_month():
    if not RUN_OPTIONAL_ROLLING_MONTH:
        print(
            "Rolling month disabled. "
            "Set RUN_OPTIONAL_ROLLING_MONTH=True only after the "
            "single-transition notebook has been validated."
        )
        return None

    dates = sorted(pd.to_datetime(quotes_all["date"].unique()))

    records = []
    previous_sigma_state = None

    for d0, d1 in tqdm(
        list(zip(dates[:-1], dates[1:])),
        desc="Rolling neural LV",
        unit="transition",
    ):
        init = (
            previous_sigma_state
            if WARM_START_SIGMA_ACROSS_DAYS
            else None
        )

        m, h, ctx, hist = train_or_load_neural_lv(
            d0, initial_sigma_state=init
        )

        if WARM_START_SIGMA_ACROSS_DAYS:
            previous_sigma_state = copy.deepcopy(m.state_dict())

        records.append({
            "date": d0,
            "next_date": d1,
            "trained_epochs": (
                int(hist["epoch"].max()) if len(hist) else np.nan
            ),
            "fixed_eval_RMSE_last": (
                hist["eval_dollar_rmse"].dropna().iloc[-1]
                if hist["eval_dollar_rmse"].notna().any()
                else np.nan
            ),
        })

    return pd.DataFrame(records)

rolling_training_log = run_optional_rolling_month()

if rolling_training_log is not None:
    display(rolling_training_log)



## 21. What to expect on the Intel-Iris-Xe laptop, and when HPC helps

### Laptop

The Intel Iris Xe is not an NVIDIA CUDA device. Unless your PyTorch installation specifically exposes an Intel `xpu` backend, this notebook runs on CPU.

That makes `pilot` useful for:

- verifying the full methodology;
- obtaining the first concrete same-day / next-day / hedge results;
- checking whether the control variate genuinely reduces variance;
- inspecting convergence;
- discovering data or modelling bugs.

### NVIDIA GPU / Imperial HPC

A GPU or HPC node becomes valuable when you want to:

- increase from hundreds to tens of thousands of paths per SGD step;
- restore 20 SGD iterations per epoch;
- run hundreds or 1000 epochs;
- use 200k-400k final Monte Carlo paths;
- repeat each daily calibration over several random initialisations;
- roll all 19 transitions;
- later compare Sabate training against Fan's unbiased-gradient training.

### What is *not* changed by moving hardware

Do not change the cleaning universe, frozen-next-day definition or hedge accounting merely because the hardware changes. Hardware should primarily buy lower Monte Carlo error and more robust optimisation.

---

## 22. References used to construct this notebook

**Primary paper**

Gierjatowicz, P., Sabate-Vidales, M., Šiška, D., Szpruch, L. and Žurič, Ž.  
*Robust Pricing and Hedging via Neural SDEs*.

Especially:

- Section 2.1: hedging strategy as a control variate;
- Section 2.2: tamed Euler;
- Section 2.3: alternating SDE and hedge optimisation;
- Section 2.4: original one-network-per-maturity incremental method;
- Sections 5.1 and 5.3: neural LV architecture and numerical settings.

**Repository**

`msabvid/robust_nsde`, especially:

- `nsde_LV.py`
- `networks.py`

**Benchmark conventions copied from the supplied project notebooks**

- `03a_LV_Raw_SVI_Efficient_Cached_v1`
- `04_Heston_COS`

The cleaning rules, identical-contract pairing, frozen one-day experiment and one-day hedge P&L convention are kept aligned with those notebooks.
